In [ ]:
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, GridSearchCV, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, root_mean_squared_error

path = "data/dataset.xlsx"
#sheet = "data" #data with no ionization descriptors
sheet = "data_ionization" #data including the ionization 

target_col = "CHI,IAM"
set_col = "Set"  
RESPECT_EXISTING_SET_SIZE = True  
TRAIN_RATIO = 0.80                
RANDOM_STATE = 42

df = pd.read_excel(path, sheet = sheet)

df[target_col] = pd.to_numeric(df[target_col], errors="coerce")

feature_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if target_col in feature_cols:
    feature_cols.remove(target_col)
feature_cols = sorted(feature_cols)

def kennard_stone(X, k):
    X = np.asarray(X, dtype=float)
    n = X.shape[0]
    if not (1 < k < n):
        raise ValueError("k has to be in the range (1, n_samples).")

    centroid = X.mean(axis=0, keepdims=True)
    d_centroid = np.linalg.norm(X - centroid, axis=1)
    first = int(np.argmax(d_centroid))

    selected_mask = np.zeros(n, dtype=bool)
    selected_mask[first] = True
    selected = [first]

    min_dists = np.linalg.norm(X - X[first], axis=1)

    while len(selected) < k:
        unselected = np.where(~selected_mask)[0]
        next_idx = unselected[np.argmax(min_dists[unselected])]
        selected.append(int(next_idx))
        selected_mask[next_idx] = True
        d_new = np.linalg.norm(X - X[next_idx], axis=1)
        min_dists = np.minimum(min_dists, d_new)

    train_idx = np.array(selected, dtype=int)
    test_idx  = np.where(~selected_mask)[0].astype(int)
    return train_idx, test_idx

imputer_split = SimpleImputer(strategy="median")
X_all = imputer_split.fit_transform(df[feature_cols].values)

scaler_split = StandardScaler()
X_all_std = scaler_split.fit_transform(X_all)

if RESPECT_EXISTING_SET_SIZE and (set_col in df.columns):
    n_train = int((df[set_col] == "T").sum())
    if n_train <= 0 or n_train >= len(df):
        n_train = int(round(TRAIN_RATIO * len(df)))
else:
    n_train = int(round(TRAIN_RATIO * len(df)))

train_idx, test_idx = kennard_stone(X_all_std, k=n_train)

train_df = df.iloc[train_idx].copy()
test_df  = df.iloc[test_idx].copy()

X_train = train_df[feature_cols].to_numpy()
y_train = train_df[target_col].to_numpy()
X_test  = test_df[feature_cols].to_numpy()
y_test  = test_df[target_col].to_numpy()

print(f"[Kennard–Stone] Train shape: {X_train.shape}, Test shape: {X_test.shape}, #features: {len(feature_cols)}")

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
SCORING_RMSE = "neg_root_mean_squared_error"

RESULTS = []

def _strip_model_prefix(d):
    out = {}
    for k, v in d.items():
        out[k.replace("model__", "", 1) if k.startswith("model__") else k] = v
    return out

def evaluate_and_collect(name, estimator, param_grid=None, scale=False, verbose=0):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", estimator))
    pipe = Pipeline(steps)

    if param_grid:
        grid = {f"model__{k}": v for k, v in param_grid.items()}
        gs = GridSearchCV(
            estimator=pipe,
            param_grid=grid,
            scoring=SCORING_RMSE,
            cv=cv,
            n_jobs=-1,
            verbose=verbose
        )
        gs.fit(X_train, y_train)
        best_pipe = gs.best_estimator_
        best_params = _strip_model_prefix(gs.best_params_)
    else:
        best_pipe = pipe.fit(X_train, y_train)
        best_params = {}

    cv_scores = cross_validate(
        best_pipe, X_train, y_train,
        scoring=["r2", SCORING_RMSE],
        cv=cv, return_train_score=False, n_jobs=-1
    )
    cv_Q2_mean = float(np.mean(cv_scores["test_r2"]))
    cv_Q2_sd   = float(np.std(cv_scores["test_r2"]))
    cv_RMSE    = float(-np.mean(cv_scores["test_neg_root_mean_squared_error"]))

    best_pipe.fit(X_train, y_train)
    pred_train = best_pipe.predict(X_train)
    pred_test  = best_pipe.predict(X_test)

    row = {
        "model": name,
        "cv_Q2_mean": float(cv_Q2_mean),
        "cv_Q2_sd": float(cv_Q2_sd),
        "cv_RMSE": float(cv_RMSE),
        "train_R2": float(r2_score(y_train, pred_train)),
        "train_RMSE": float(root_mean_squared_error(y_train, pred_train)),
        "test_R2": float(r2_score(y_test, pred_test)),
        "test_RMSE": float(root_mean_squared_error(y_test, pred_test)),
        "best_params": json.dumps(best_params)
    }

    print("\n=== Results:", name, "===")
    for k in ["cv_Q2_mean","cv_Q2_sd","cv_RMSE","train_R2","train_RMSE","test_R2","test_RMSE"]:
        print(f"{k}: {row[k]:.6f}")
    print("best_params:", row["best_params"])

    RESULTS.append(row)
    return row

if RESULTS:
    summary_df = pd.DataFrame(RESULTS).sort_values(by="test_R2", ascending=False).reset_index(drop=True)
    cols = ["model","cv_Q2_mean","cv_Q2_sd","cv_RMSE","test_R2","test_RMSE","train_R2","train_RMSE","best_params"]
    print("\n=== SUMMARY (sorted by Test R²) ===")
    print(summary_df[cols].to_string(index=False))

[Kennard–Stone] Train shape: (794, 9), Test shape: (199, 9), #features: 9


In [18]:
# Saving the file with split
from pathlib import Path

df_with_set = df.copy()
df_with_set[set_col] = "Test"
df_with_set.loc[train_idx, set_col] = "Train"

in_path = Path(path)
xlsx_out = in_path.with_name(in_path.stem + "_with_split.xlsx")

df_with_set.to_excel(xlsx_out, index=False)

print(f"Saved split with column '{set_col}':")
print(f"Train n={len(train_idx)}, Test n={len(test_idx)}")

Saved split with column 'Set':
Train n=794, Test n=199


In [19]:
# MLR

from sklearn.linear_model import LinearRegression

mlr = LinearRegression()
mlr_grid = {
    "fit_intercept": [True, False],
    "positive": [False, True],
}

evaluate_and_collect("MLR", mlr, param_grid=mlr_grid, scale=True, verbose=0)


=== Results: MLR ===
cv_Q2_mean: 0.751850
cv_Q2_sd: 0.035789
cv_RMSE: 8.043704
train_R2: 0.762180
train_RMSE: 7.942887
test_R2: 0.667922
test_RMSE: 6.930521
best_params: {"fit_intercept": true, "positive": false}


{'model': 'MLR',
 'cv_Q2_mean': 0.7518499009546123,
 'cv_Q2_sd': 0.035788876962206465,
 'cv_RMSE': 8.043704418709167,
 'train_R2': 0.7621797412798142,
 'train_RMSE': 7.942887180743726,
 'test_R2': 0.6679218219322659,
 'test_RMSE': 6.930520968961914,
 'best_params': '{"fit_intercept": true, "positive": false}'}

In [20]:
# SVR linear

from sklearn.svm import SVR

svr_lin = SVR(kernel="linear")
svr_lin_grid = {
    "C": [0.1, 1, 10],
    "epsilon": [0.01, 0.1, 0.5],
}

evaluate_and_collect("SVR (linear)", svr_lin, param_grid=svr_lin_grid, scale=True, verbose=0)


=== Results: SVR (linear) ===
cv_Q2_mean: 0.741608
cv_Q2_sd: 0.032102
cv_RMSE: 8.217707
train_R2: 0.755636
train_RMSE: 8.051420
test_R2: 0.681477
test_RMSE: 6.787597
best_params: {"C": 10, "epsilon": 0.5}


{'model': 'SVR (linear)',
 'cv_Q2_mean': 0.7416082658643205,
 'cv_Q2_sd': 0.03210245566089245,
 'cv_RMSE': 8.217706661935834,
 'train_R2': 0.7556361268386618,
 'train_RMSE': 8.051419782999714,
 'test_R2': 0.6814771352873197,
 'test_RMSE': 6.787596538937699,
 'best_params': '{"C": 10, "epsilon": 0.5}'}

In [21]:
# SVR-RBF 

from sklearn.svm import SVR

svr_rbf = SVR(kernel="rbf")
svr_rbf_grid = {
    "C": [1, 10, 100],
    "gamma": ["scale", 0.01],
    "epsilon": [0.01, 0.1],
}

evaluate_and_collect("SVR (RBF)", svr_rbf, param_grid=svr_rbf_grid, scale=True, verbose=0)


=== Results: SVR (RBF) ===
cv_Q2_mean: 0.811323
cv_Q2_sd: 0.048167
cv_RMSE: 6.989209
train_R2: 0.884055
train_RMSE: 5.546010
test_R2: 0.853148
test_RMSE: 4.608779
best_params: {"C": 100, "epsilon": 0.1, "gamma": "scale"}


{'model': 'SVR (RBF)',
 'cv_Q2_mean': 0.811323030861894,
 'cv_Q2_sd': 0.04816650363567116,
 'cv_RMSE': 6.989209240713319,
 'train_R2': 0.8840546272752733,
 'train_RMSE': 5.546010415414431,
 'test_R2': 0.8531478385638096,
 'test_RMSE': 4.60877901289097,
 'best_params': '{"C": 100, "epsilon": 0.1, "gamma": "scale"}'}

In [22]:
# Random Forest 
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(random_state=42, n_jobs=-1)
rf_grid = {
    "n_estimators": [300, 600],
    "max_depth": [None, 8, 16],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", 0.6],
    "bootstrap": [True],
}

evaluate_and_collect("RandomForest", rf, param_grid=rf_grid, scale=False, verbose=0)


=== Results: RandomForest ===
cv_Q2_mean: 0.778474
cv_Q2_sd: 0.027695
cv_RMSE: 7.619347
train_R2: 0.970911
train_RMSE: 2.777923
test_R2: 0.850611
test_RMSE: 4.648420
best_params: {"bootstrap": true, "max_depth": null, "max_features": 0.6, "min_samples_leaf": 1, "min_samples_split": 2, "n_estimators": 300}


{'model': 'RandomForest',
 'cv_Q2_mean': 0.7784735036933157,
 'cv_Q2_sd': 0.027695369564821364,
 'cv_RMSE': 7.619346684337837,
 'train_R2': 0.9709107447681609,
 'train_RMSE': 2.7779234342400483,
 'test_R2': 0.8506107449343282,
 'test_RMSE': 4.648420350838736,
 'best_params': '{"bootstrap": true, "max_depth": null, "max_features": 0.6, "min_samples_leaf": 1, "min_samples_split": 2, "n_estimators": 300}'}

In [23]:
# Decision Tree 

from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(random_state=42)
dt_grid = {
    "max_depth": [None, 8, 16],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": [None, "sqrt", 0.6],
}

evaluate_and_collect("DecisionTree", dt, param_grid=dt_grid, scale=False, verbose=0)


=== Results: DecisionTree ===
cv_Q2_mean: 0.633917
cv_Q2_sd: 0.044114
cv_RMSE: 9.800804
train_R2: 0.890870
train_RMSE: 5.380534
test_R2: 0.780762
test_RMSE: 5.631233
best_params: {"max_depth": null, "max_features": 0.6, "min_samples_leaf": 4, "min_samples_split": 2}


{'model': 'DecisionTree',
 'cv_Q2_mean': 0.633916909267282,
 'cv_Q2_sd': 0.04411392522616871,
 'cv_RMSE': 9.800803937317173,
 'train_R2': 0.8908703441149136,
 'train_RMSE': 5.380533791099299,
 'test_R2': 0.7807620943354203,
 'test_RMSE': 5.631233494599074,
 'best_params': '{"max_depth": null, "max_features": 0.6, "min_samples_leaf": 4, "min_samples_split": 2}'}

In [27]:
# LightGBM

from lightgbm import LGBMRegressor


lgbm = LGBMRegressor(
    random_state=42,
    learning_rate=0.03,
    subsample_freq=1,
    n_jobs=-1,
    verbose=-1
)

lgbm_grid = {
    "n_estimators":      [800, 1600],
    "num_leaves":        [8, 16],
    "max_depth":         [4, 6],
    "min_child_samples": [40, 100],
    "subsample":         [0.7],       
    "colsample_bytree":  [0.7, 1.0],
    "reg_alpha":         [0.0, 0.5],
    "reg_lambda":        [1.0, 5.0],
}

evaluate_and_collect("LightGBM", lgbm, param_grid=lgbm_grid, scale=False, verbose=0)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/skle


=== Results: LightGBM ===
cv_Q2_mean: 0.777470
cv_Q2_sd: 0.047756
cv_RMSE: 7.605188
train_R2: 0.927572
train_RMSE: 4.383367
test_R2: 0.809027
test_RMSE: 5.255715
best_params: {"colsample_bytree": 1.0, "max_depth": 4, "min_child_samples": 40, "n_estimators": 1600, "num_leaves": 16, "reg_alpha": 0.5, "reg_lambda": 5.0, "subsample": 0.7}


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


{'model': 'LightGBM',
 'cv_Q2_mean': 0.777470499623056,
 'cv_Q2_sd': 0.04775564321144933,
 'cv_RMSE': 7.605187939487202,
 'train_R2': 0.9275718104281887,
 'train_RMSE': 4.383366509638007,
 'test_R2': 0.8090269095446242,
 'test_RMSE': 5.255714931040827,
 'best_params': '{"colsample_bytree": 1.0, "max_depth": 4, "min_child_samples": 40, "n_estimators": 1600, "num_leaves": 16, "reg_alpha": 0.5, "reg_lambda": 5.0, "subsample": 0.7}'}

In [25]:
# kNN

from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline as SKPipeline 

n_train = X_train.shape[0]
k_grid = sorted({
    5, 7, 9, 11, 15, 21, 31, 41,
    max(5, int(n_train**0.5)),          
    max(5, int(0.05 * n_train)),        
    max(5, int(0.10 * n_train)),       
})
k_grid = [k for k in k_grid if k < n_train]  

knn_with_pca = SKPipeline(steps=[
    ("pca", PCA(random_state=RANDOM_STATE)),
    ("knn", KNeighborsRegressor())
])



knn_grid = {
    "pca__n_components": [None, 0.95, 0.90, 0.85, min(100, X_train.shape[1] // 2)],
    "knn__n_neighbors": k_grid,
    "knn__weights": ["uniform"],
    "knn__p": [1, 2],
    "knn__leaf_size": [20, 30, 40],
    "knn__metric": ["minkowski"], 
}

evaluate_and_collect("KNN", knn_with_pca, param_grid=knn_grid, scale=True, verbose=1)

Fitting 5 folds for each of 330 candidates, totalling 1650 fits

=== Results: KNN ===
cv_Q2_mean: 0.697214
cv_Q2_sd: 0.034878
cv_RMSE: 8.906041
train_R2: 0.815640
train_RMSE: 6.993385
test_R2: 0.829401
test_RMSE: 4.967453
best_params: {"knn__leaf_size": 20, "knn__metric": "minkowski", "knn__n_neighbors": 5, "knn__p": 2, "knn__weights": "uniform", "pca__n_components": null}


{'model': 'KNN',
 'cv_Q2_mean': 0.6972140330848607,
 'cv_Q2_sd': 0.034878224525795594,
 'cv_RMSE': 8.906041239693298,
 'train_R2': 0.8156398999360039,
 'train_RMSE': 6.993385003801701,
 'test_R2': 0.829401154151805,
 'test_RMSE': 4.967452893762102,
 'best_params': '{"knn__leaf_size": 20, "knn__metric": "minkowski", "knn__n_neighbors": 5, "knn__p": 2, "knn__weights": "uniform", "pca__n_components": null}'}

In [26]:
if not RESULTS:
    print("No results.")
else:
    summary_df = pd.DataFrame(RESULTS)
    summary_df_sorted = summary_df.sort_values(by="test_R2", ascending=False).reset_index(drop=True)

    print("\n=== SUMMARY (sorted by Test R²) ===")
    cols = ["model","cv_Q2_mean","cv_Q2_sd","cv_RMSE","test_R2","test_RMSE","train_R2","train_RMSE","best_params"]
    print(summary_df_sorted[cols].to_string(index=False))



=== SUMMARY (sorted by Test R²) ===
       model  cv_Q2_mean  cv_Q2_sd  cv_RMSE  test_R2  test_RMSE  train_R2  train_RMSE                                                                                                                                                       best_params
   SVR (RBF)    0.811323  0.048167 6.989209 0.853148   4.608779  0.884055    5.546010                                                                                                                      {"C": 100, "epsilon": 0.1, "gamma": "scale"}
RandomForest    0.778474  0.027695 7.619347 0.850611   4.648420  0.970911    2.777923                                   {"bootstrap": true, "max_depth": null, "max_features": 0.6, "min_samples_leaf": 1, "min_samples_split": 2, "n_estimators": 300}
         KNN    0.697214  0.034878 8.906041 0.829401   4.967453  0.815640    6.993385                      {"knn__leaf_size": 20, "knn__metric": "minkowski", "knn__n_neighbors": 5, "knn__p": 2, "knn__weights": "uniform"